# 01_chat_sse_frontend

Stage 1 of the `codes/fastapi/` track: **FastAPI + Pydantic + `Depends` + async
Server-Sent Events + a no-build static frontend**.

Notebook path: `codes/fastapi/01_chat_sse_frontend/qa.ipynb`. Deps come from the
shared `codes/requirements.txt` (`fastapi`, `uvicorn`, `httpx`). The "LLM" is a
deterministic mock, so no API key is needed.

Shape to keep in your head:

`browser (static/index.html)` → `POST /api/chat/stream` (Pydantic `ChatRequest`)
→ `Depends(get_backend)` → `async for token` → `StreamingResponse` of SSE frames
→ `fetch` + `ReadableStream` in the page.

Each exercise below gives the question first, then a `# answer` cell that has
been run. Top-level `await` works because Jupyter owns the event loop.

In [1]:
from pathlib import Path
import socket
import subprocess
import sys
import time

def lab_root() -> Path:
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "01_chat_sse_frontend" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "fastapi" / "01_chat_sse_frontend"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    raise FileNotFoundError("cannot locate codes/fastapi/01_chat_sse_frontend")

ROOT = lab_root()
sys.path.insert(0, str(ROOT))
print(ROOT.relative_to(ROOT.parents[2]))

import httpx

with socket.socket() as s:  # ask the OS for a free port so reruns never collide
    s.bind(("127.0.0.1", 0))
    PORT = s.getsockname()[1]
BASE = f"http://127.0.0.1:{PORT}"
_server = subprocess.Popen(
    [sys.executable, str(ROOT / "app.py"), "--port", str(PORT)],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
for _ in range(50):
    try:
        health = httpx.get(f"{BASE}/api/health", timeout=1.0).json()
        break
    except httpx.TransportError:
        time.sleep(0.1)
print("server ready →", health)

codes/fastapi/01_chat_sse_frontend


server ready → {'status': 'ok', 'backend': 'mock'}


## 1. Routes → OpenAPI

FastAPI builds `/openapi.json` from the route decorators and the Pydantic
models. List every path with its HTTP methods, hiding the docs/static routes.

In [2]:
# answer
spec = httpx.get(f"{BASE}/openapi.json").json()
for path, ops in sorted(spec["paths"].items()):
    print(f"{path:<20}", sorted(m.upper() for m in ops))
print("schemas →", sorted(spec["components"]["schemas"]))

/api/chat            ['POST']
/api/chat/stream     ['POST']
/api/health          ['GET']
/api/stats           ['GET']
schemas → ['ChatRequest', 'ChatResponse', 'HTTPValidationError', 'Message', 'Stats', 'ValidationError']


## 2. Pydantic validation is the 422 for free

Send `/api/chat` an empty `content`, then an unknown `role`. Print the status
and each error's `loc` + `msg`. Neither case needs an `if` in the route.

In [3]:
# answer
bad_bodies = [
    {"messages": [{"role": "user", "content": ""}]},
    {"messages": [{"role": "robot", "content": "hi"}]},
    {"messages": []},
]
for body in bad_bodies:
    r = httpx.post(f"{BASE}/api/chat", json=body)
    print(r.status_code, [(e["loc"], e["msg"]) for e in r.json()["detail"]])

422 [(['body', 'messages', 0, 'content'], 'String should have at least 1 character')]
422 [(['body', 'messages', 0, 'role'], "Input should be 'system', 'user' or 'assistant'")]
422 [(['body', 'messages'], 'List should have at least 1 item after validation, not 0')]


## 3. SSE is just text — read the raw bytes

Open `/api/chat/stream` and print the first three **raw** frames
(frames) exactly as sent, using `iter_lines`. Then print the final frame.
A frame is `event:` + `data:` + a blank line.

In [4]:
# answer
BODY = {"messages": [{"role": "user", "content": "explain SSE"}]}
frames, cur = [], []
with httpx.stream("POST", f"{BASE}/api/chat/stream", json=BODY) as resp:
    print("content-type →", resp.headers["content-type"])
    for line in resp.iter_lines():
        if line == "":
            frames.append(cur)
            cur = []
        else:
            cur.append(line)
for f in frames[:3]:
    print(f)
print("...")
print(frames[-1], "| total frames:", len(frames))

content-type → text/event-stream; charset=utf-8


['event: token', 'data: {"text": "You "}']
['event: token', 'data: {"text": "said: "}']
['event: token', 'data: {"text": "\'explain "}']
...
['event: done', 'data: {"tokens": 23}'] | total frames: 24


## 4. Streaming vs blocking: time to first token

`/api/chat` returns after the last token; `/api/chat/stream` emits the first one
almost immediately. Measure both with `time.perf_counter`.

In [5]:
# answer
t0 = time.perf_counter()
one = httpx.post(f"{BASE}/api/chat", json=BODY).json()
t_block = time.perf_counter() - t0

t0 = time.perf_counter()
first = None
with httpx.stream("POST", f"{BASE}/api/chat/stream", json=BODY) as resp:
    for line in resp.iter_lines():
        if line.startswith("event: token") and first is None:
            first = time.perf_counter() - t0
total = time.perf_counter() - t0
print(f"blocking  : {t_block:.2f}s until anything is usable")
print(f"streaming : first token {first:.2f}s, full reply {total:.2f}s ({one['tokens']} tokens)")
assert first < total / 3

blocking  : 0.71s until anything is usable
streaming : first token 0.04s, full reply 0.72s (23 tokens)


## 5. `Depends` + `dependency_overrides`: swap the backend in a test

Route handlers take `backend: Backend` and never build one. Override
`get_backend` with a fake that yields exactly `["a", "b", "c"]`, call the app
**in-process** with `httpx.ASGITransport` (no server), and print the reply.
Remove the override afterwards.

(`ASGITransport` buffers the body, so it is good for result tests, not for
timing a stream — that is why exercise 4 uses the real server.)

In [6]:
# answer
import app as lab

class FakeBackend:
    async def stream(self, messages):
        for t in ["a", "b", "c"]:
            yield t

lab.app.dependency_overrides[lab.get_backend] = lambda: FakeBackend()
try:
    transport = httpx.ASGITransport(app=lab.app)
    async with httpx.AsyncClient(transport=transport, base_url="http://test") as http:
        r = await http.post("/api/chat", json=BODY)
    print(r.json())
finally:
    lab.app.dependency_overrides.clear()
print("overrides left →", lab.app.dependency_overrides)

{'reply': 'abc', 'tokens': 3}
overrides left → {}


## 6. An error after the headers: in-band `error` event

Once the first frame is out, the status line (200) is already sent, so an
exception cannot become a 500. Override the backend with one that yields a
token and then raises; call the stream route in-process and print the status
and the body. Then print `STATS` to see `failed` incremented.

In [7]:
# answer
class FlakyBackend:
    async def stream(self, messages):
        yield "ok "
        raise RuntimeError("upstream 500")

lab.app.dependency_overrides[lab.get_backend] = lambda: FlakyBackend()
try:
    transport = httpx.ASGITransport(app=lab.app)
    async with httpx.AsyncClient(transport=transport, base_url="http://test") as http:
        r = await http.post("/api/chat/stream", json=BODY)
    print(r.status_code)
    print(r.text)
finally:
    lab.app.dependency_overrides.clear()
print(lab.STATS)

200
event: token
data: {"text": "ok "}

event: error
data: {"message": "RuntimeError: upstream 500"}


Stats(started=1, completed=0, cancelled=0, failed=1, active=0)


## 7. A closed connection cancels the generator

Read two frames from the stream and leave the `with` block (the connection
closes mid-reply). Poll `/api/stats` until `cancelled` goes up by one. This is
the same `asyncio.CancelledError` the concurrency track teaches — here it is
triggered by the browser's **Stop** button (`AbortController.abort()`).

In [8]:
# answer
before = httpx.get(f"{BASE}/api/stats").json()
with httpx.stream("POST", f"{BASE}/api/chat/stream", json=BODY) as resp:
    for i, _line in enumerate(resp.iter_lines()):
        if i >= 3:        # 2 complete frames + a bit
            break
for _ in range(40):
    after = httpx.get(f"{BASE}/api/stats").json()
    if after["cancelled"] > before["cancelled"]:
        break
    time.sleep(0.1)
print("before →", before)
print("after  →", after)
assert after["cancelled"] == before["cancelled"] + 1 and after["active"] == 0

before → {'started': 2, 'completed': 2, 'cancelled': 0, 'failed': 0, 'active': 0}
after  → {'started': 3, 'completed': 2, 'cancelled': 1, 'failed': 0, 'active': 0}


## 8. Swap in a real-LLM backend (offline proof)

`OpenAICompatBackend` speaks the OpenAI `chat/completions` SSE format over
`httpx`. Prove it without a network or key: feed it an `httpx.MockTransport`
that replies with three `data:` chunks and `[DONE]`, then collect its tokens.
The route code would not change — only `get_backend` picks a different class.

In [9]:
# answer
import json

def fake_openai(request: httpx.Request) -> httpx.Response:
    sent = json.loads(request.content)
    assert sent["stream"] is True and sent["messages"][0]["content"] == "explain SSE"
    chunks = [{"choices": [{"delta": {"content": t}}]} for t in ["Hel", "lo", "!"]]
    chunks.insert(0, {"choices": [{"delta": {"role": "assistant"}}]})  # no content: must be skipped
    body = "".join(f"data: {json.dumps(c)}\n\n" for c in chunks) + "data: [DONE]\n\n"
    return httpx.Response(200, content=body.encode(), headers={"content-type": "text/event-stream"})

settings = lab.Settings(backend="openai", openai_api_key="sk-test", openai_base_url="http://fake/v1")
backend = lab.OpenAICompatBackend(settings, transport=httpx.MockTransport(fake_openai))
tokens = [t async for t in backend.stream([lab.Message(role="user", content="explain SSE")])]
print(tokens)
assert "".join(tokens) == "Hello!"

['Hel', 'lo', '!']


## 9. Open the page

The frontend is `static/index.html`: plain HTML + `fetch` + a ~25-line SSE
parser, no build step. In a terminal run `python app.py` and open
http://127.0.0.1:8000/ . Here, just confirm the server hands the page out and
that it calls the streaming endpoint.

In [10]:
# answer
page = httpx.get(f"{BASE}/")
print(page.status_code, page.headers["content-type"], len(page.text), "bytes")
print("uses fetch →", "fetch(\"/api/chat/stream\"" in page.text)
print("has Stop/abort →", "AbortController" in page.text)

200 text/html; charset=utf-8 5959 bytes
uses fetch → True
has Stop/abort → True


In [11]:
_server.terminate()
_server.wait(timeout=5)
print("server stopped, exit code", _server.returncode)

server stopped, exit code -15
